# PID tuning in practice: notebook

The running example of the Note: the chapter's robot (8 kg, drag 2 N·s/m, drive force ±4 N; yaw inertia 0.1 kg·m², torque ±0.2 N·m) controlled at 100 Hz. The speed loop sees the average of the last 20 wheel speeds; the speed PI has RO-060's gains kp = 20, ki = 10; the heading loop is RO-060's PID with kp = 0.2, ki = 0.05, kd = 0.283. Every number in the Note is computed below. The simulation is deterministic (no random numbers).

In [1]:
import numpy as np
import pandas as pd
from scipy import signal

# the chapter's robot (RO-07): speed plant m dv/dt = u - b v + d, heading plant I dw/dt = torque
M, B, U_MAX = 8.0, 2.0, 4.0          # kg, N s/m, N
I_YAW, TAU_MAX = 0.1, 0.2            # kg m^2, N m
DT = 0.01                            # s, the controller runs at 100 Hz
KP, KI = 20.0, 10.0                  # the speed PI of RO-060

NW = 20                                  # samples in the speed average (0.2 s)
G = 9.81
RAMP = M * G * np.sin(np.radians(1.0))   # 1.37 N pulling back on a 1 degree ramp
KC, TC = 200.0, 0.40                     # critical gain (N per m/s) and period (s) read off the experiment
ZN_PI = (0.45 * KC, 0.45 * KC / (TC / 1.2))                      # 90, 270
K_FOTD, TAU_FOTD, T_FOTD = 0.5, 0.094, 4.00                      # bump-test model
FBS_PI = ((0.15 * TAU_FOTD + 0.35 * T_FOTD) / (K_FOTD * TAU_FOTD),
          (0.46 * TAU_FOTD + 0.02 * T_FOTD) / (K_FOTD * TAU_FOTD ** 2))   # 30.1, 27.9
GAINS = {"RO-060's gains": (KP, KI), "bump-test rule": FBS_PI, "Ziegler–Nichols": ZN_PI}
HKP, HKI, HKD = 0.2, 0.05, 0.283         # heading PID of RO-060
TD = HKD / HKP                           # derivative time 0.5 s
R_WHEEL, L_WHEELS, TICKS = 0.05, 0.2, 1024
DS = 2 * np.pi * R_WHEEL / TICKS         # 0.307 mm of wheel travel per encoder tick
F_VIB, A_VIB = 10.0, 0.3                 # frame bending: 10 Hz, 0.3 rad/s on the gyro


def speed_run(T, kp, ki, *, r=lambda t: 0.5, d=lambda t: 0.0, v0=0.5, bias=None, extra=0, nw=NW):
    """PI (or P with a fixed bias force) on the averaged speed. extra: samples of added delay. Returns t, v, y, u."""
    n = int(round(T / DT)) + 1
    t = np.arange(n) * DT
    v, buf = v0, [v0] * (nw + extra)
    integ = B * v0
    V, Y, U = np.zeros(n), np.zeros(n), np.zeros(n)
    for k in range(n):
        y = np.mean(buf[len(buf) - nw - extra:len(buf) - extra])
        e = r(t[k]) - y
        c = (bias + kp * e) if bias is not None else kp * e + integ
        u = float(np.clip(c, -U_MAX, U_MAX))
        V[k], Y[k], U[k] = v, y, u
        integ += ki * e * DT
        v += DT * (u - B * v + d(t[k])) / M
        buf.append(v)
    return t, V, Y, U


def bump_test(T=30.0, force=1.0):
    """Open loop from rest: the drive force steps to 1 N at t = 0. Returns t, true speed, averaged speed."""
    n = int(round(T / DT)) + 1
    t = np.arange(n) * DT
    v, buf, V, Y = 0.0, [0.0] * NW, np.zeros(n), np.zeros(n)
    for k in range(n):
        V[k], Y[k] = v, np.mean(buf[-NW:])
        v += DT * (force - B * v) / M
        buf.append(v)
    return t, V, Y


def fotd_fit(t, y, force=1.0):
    """Read the bump test as Feedback Systems Fig. 11.7a: steepest tangent, its intercepts, T63 and K."""
    K = y[-1] / force
    sl = np.gradient(y, t)
    j = int(np.argmax(sl))
    tau = t[j] - y[j] / sl[j]
    a = sl[j] * t[j] - y[j]
    t63 = t[int(np.argmax(y >= 0.632 * y[-1]))]
    return dict(K=K, slope=sl[j], tj=t[j], tau=tau, a=a, T63=t63, T=t63 - tau)


def speed_loop_tf(kp, ki, w, extra=0):
    """Loop transfer L(e^{i w h}) of the 100 Hz speed loop (plant, 20-sample average, PI)."""
    z = np.exp(1j * w * DT)
    P = (DT / M) / (z - (1 - DT * B / M))
    avg = sum(z ** (-i) for i in range(NW)) / NW * z ** (-extra)
    return (kp + ki * DT / (z - 1)) * P * avg


def margins(L, w):
    """Phase margin (deg), gain crossover (rad/s), delay margin (s), stability margin min|1 + L|."""
    ig = int(np.argmin(abs(abs(L) - 1)))
    pm = np.angle(-L[ig])                    # 180 degrees plus the phase of L, in (-180, 180]
    return dict(pm=np.degrees(pm), wgc=w[ig], dm=pm / w[ig], sm=np.min(abs(1 + L)))


def lowpass(fc):
    """First-order low-pass at 100 Hz (backward difference, Feedback Systems eq. 11.18 form): (b, a)."""
    tf = 1 / (2 * np.pi * fc)
    return np.array([DT / (tf + DT)]), np.array([1.0, -tf / (tf + DT)])


def notch(f0=F_VIB, bw=5.0):
    """Second-order notch at f0 Hz, width bw Hz, at 100 Hz: (b, a)."""
    return signal.iirnotch(f0, f0 / bw, fs=1 / DT)


def heading_run(T, *, step=0.1, t_step=0.5, beta=1.0, gamma=1.0, N=None, quant=False, gyro=False, vib=False,
                filt=None, drive=0.5, phase=0.37):
    """Heading PID at 100 Hz. Derivative: on the error (gamma = 1) or on the measured heading (gamma = 0), by
    differencing the measured heading, filtered with Tf = Td/N when N is given; or from the gyro (gyro=True), with
    the frame's 10 Hz bending added (vib=True) and a filter (b, a) on the gyro. quant: the heading comes from
    1024-tick wheel encoders while driving forward at `drive` m/s. Returns a dict of arrays."""
    n = int(round(T / DT)) + 1
    t = np.arange(n) * DT
    th = w = integ = D = 0.0
    yprev = rprev = None
    s = 0.0
    zi = None if filt is None else signal.lfilter_zi(*filt) * 0.0
    keys = ("th", "y", "r", "c", "u", "P", "I", "D", "g")
    out = {k: np.zeros(n) for k in keys}
    y0 = None
    for k in range(n):
        r = step if t[k] >= t_step else 0.0
        if quant:
            nr = np.floor((s + L_WHEELS * th / 2) / DS)
            nl = np.floor((s - L_WHEELS * th / 2) / DS + phase)
            y = (nr - nl) * DS / L_WHEELS
            y0 = y if y0 is None else y0
            y -= y0
        else:
            y = th
        if yprev is None:
            yprev, rprev = y, r
        e = r - y
        P = HKP * (beta * r - y)
        g = 0.0
        if gyro:
            g = w + (A_VIB * np.sin(2 * np.pi * F_VIB * t[k]) if vib else 0.0)
            if filt is not None:
                gf, zi = signal.lfilter(*filt, [g], zi=zi)
                g = gf[0]
            D = -HKD * g
        else:
            dd = gamma * (r - rprev) - (y - yprev)
            if N:
                tf = TD / N
                D = tf / (tf + DT) * D + HKD / (tf + DT) * dd
            else:
                D = HKD * dd / DT
        c = P + integ + D
        u = float(np.clip(c, -TAU_MAX, TAU_MAX))
        for key, val in zip(keys, (th, y, r, c, u, P, integ, D, g)):
            out[key][k] = val
        integ += HKI * e * DT
        yprev, rprev = y, r
        w += DT * u / I_YAW
        th += DT * w
        s += drive * DT
    out["t"] = t
    return out


def heading_loop_tf(w, *, N=None, gyro_filt=None):
    """Loop transfer at the plant input of the 100 Hz heading PID (derivative on measurement)."""
    z = np.exp(1j * w * DT)
    W = (DT / I_YAW) / (z - 1)                 # torque -> turn rate (w[k+1] = w[k] + h u/I)
    TH = DT * z * W / (z - 1)                  # torque -> heading (th[k+1] = th[k] + h w[k+1])
    pi = HKP + HKI * DT / (z - 1)
    if gyro_filt is not None:
        b, a = gyro_filt
        F = np.polyval(b[::-1], 1 / z) / np.polyval(a[::-1], 1 / z)
        return pi * TH + HKD * F * W
    if N:
        tf = TD / N
        dpath = HKD / (tf + DT) * (1 - 1 / z) / (1 - tf / (tf + DT) / z)
    else:
        dpath = HKD * (1 - 1 / z) / DT
    return (pi + dpath) * TH


def overshoot(o, step=0.1):
    return (o["th"].max() - step) / step * 100


def settle(o, step=0.1, band=0.02, t_step=0.5):
    out = np.where(abs(o["th"] - step) > band * step)[0]
    return o["t"][out[-1]] - t_step if len(out) else 0.0




## 1. The two PID forms (Note, Section 2)

In [2]:
kp, Ti, Td = 0.2, 4.0, 1.415
print("heading PID: ki =", kp / Ti, " kd =", kp * Td, " (Ti = kp/ki =", HKP / HKI, ", Td = kd/kp =", HKD / HKP, ")")
print("speed PI: Ti =", KP / KI, "s")
print("integral time check:", KP * 0.1, "N now;", KI * 0.1, "N per s;", KI * 0.1 * KP / KI, "N after Ti")
print("derivative time check:", round(HKD * -0.1, 4), "N m now;", round(HKP * -0.1 * TD, 4), "N m change of P over Td")

heading PID: ki = 0.05  kd = 0.28300000000000003  (Ti = kp/ki = 4.0 , Td = kd/kp = 1.4149999999999998 )
speed PI: Ti = 2.0 s
integral time check: 2.0 N now; 1.0 N per s; 2.0 N after Ti
derivative time check: -0.0283 N m now; -0.0283 N m change of P over Td


## 2. The ideal model never oscillates; the average is late (Section 3.1 and 3.2)

In [3]:
for kp_ in (20, 200):
    print(f"kp = {kp_}: time constant m/(kp + b) = {M / (kp_ + B):.4f} s")
lag = (NW - 1) / 2 * DT
print("averaging lag", lag, "s; speed gap at 0.25 m/s^2:", round(0.25 * lag, 4), "m/s")

kp = 20: time constant m/(kp + b) = 0.3636 s
kp = 200: time constant m/(kp + b) = 0.0396 s
averaging lag 0.095 s; speed gap at 0.25 m/s^2: 0.0238 m/s


## 3. The Ziegler–Nichols experiment (Section 3.3 and 3.4)

RO-059's speed loop (1 N plus kp times the error) with kp raised from 20: proportional control only, bias 1 N, command step 0.5 → 0.51 m/s at t = 0.2 s.

In [4]:
def peaks(t, y):
    return [i for i in range(1, len(y) - 1) if y[i] > y[i - 1] and y[i] >= y[i + 1] and t[i] > 1.0]

for kp_ in (50, 100, 150, 200, 220):
    t, v, y, u = speed_run(3.0, kp_, 0, r=lambda t: 0.51 if t >= 0.2 else 0.5, bias=1.0)
    pk = peaks(t, y)
    amp = y[pk] - 0.5
    print(f"kp = {kp_}: periods {np.round(np.diff(t[pk]), 2)}, peak heights {np.round(amp, 4)}, force {u.min():.2f} to {u.max():.2f} N")

kp = 50: periods [0.73 0.72], peak heights [0.0096 0.0096 0.0096], force 0.96 to 1.50 N
kp = 100: periods [0.5 0.5], peak heights [0.0101 0.0099 0.0098], force 0.60 to 2.00 N
kp = 150: periods [0.43 0.44 0.43], peak heights [0.0122 0.0113 0.0108 0.0104], force 0.07 to 2.50 N
kp = 200: periods [0.4  0.39 0.4  0.39], peak heights [0.0176 0.0175 0.0175 0.0174 0.0174], force -0.54 to 3.00 N
kp = 220: periods [0.38 0.39 0.39 0.39], peak heights [0.0206 0.0221 0.0239 0.0254 0.0263], force -2.58 to 4.00 N


In [5]:
w = np.linspace(15, 16.5, 300001)
z = np.exp(1j * w * DT)
P = (DT / M) / (z - (1 - DT * B / M))
A = sum(z ** (-i) for i in range(NW)) / NW
i = np.argmin(abs(np.angle(P * A) + np.pi))
print(f"critical frequency {w[i]:.3f} rad/s, period {2 * np.pi / w[i]:.3f} s, critical gain {1 / abs(P[i] * A[i]):.1f}")
print(f"phase: drive (continuous) {-np.degrees(np.arctan(4 * w[i])):.1f}, average {np.degrees(np.angle(A[i])):.1f}, "
      f"drive at 100 Hz {np.degrees(np.angle(P[i])):.1f} deg")
print("Ziegler-Nichols PI: kp =", 0.45 * KC, " Ti =", round(TC / 1.2, 3), " ki =", 0.45 * KC / (TC / 1.2))

critical frequency 15.865 rad/s, period 0.396 s, critical gain 200.7
phase: drive (continuous) -89.1, average -86.4, drive at 100 Hz -93.6 deg
Ziegler-Nichols PI: kp = 90.0  Ti = 0.333  ki = 269.99999999999994


## 4. The bump test and the model-based rule (Section 4.1 and 4.2)

In [6]:
t, v, y = bump_test()
f = fotd_fit(t, y)
print({k: round(float(x), 4) for k, x in f.items()})
K, tau, T = 0.5, 0.094, 4.00
print("FBS 11.10a: kp =", round((0.15 * tau + 0.35 * T) / (K * tau), 2), " ki =", round((0.46 * tau + 0.02 * T) / (K * tau ** 2), 2))
a = 0.0115
print("Ziegler-Nichols step method: kp =", round(0.9 / a, 1), " Ti =", round(tau / 0.3, 3), " ki =", round(0.9 / a / (tau / 0.3), 1))
print("normalised delay", round(tau / (T + tau), 3))

{'K': 0.4997, 'slope': 0.1219, 'tj': 0.2, 'tau': 0.094, 'a': 0.0115, 'T63': 4.09, 'T': 3.996}
FBS 11.10a: kp = 30.09  ki = 27.89
Ziegler-Nichols step method: kp = 78.3  Ti = 0.313  ki = 249.8
normalised delay 0.023


## 5. Three tunings side by side (Section 4.3)

The 1° ramp from t = 1 s; margins from the 100 Hz loop; a 0.05 m/s step of the command; the same loops with 4 more samples of delay.

In [7]:
print("1 degree ramp:", round(RAMP, 3), "N")
wg = np.logspace(-2, np.log10(np.pi / DT * 0.999), 200000)
rows = []
for name, (kp_, ki_) in GAINS.items():
    m = margins(speed_loop_tf(kp_, ki_, wg), wg)
    t, v, y, u = speed_run(14, kp_, ki_, d=lambda t: -RAMP if t >= 1 else 0.0)
    out = np.where(abs(v - 0.5) > 0.005)[0]
    t2, v2, y2, u2 = speed_run(10, kp_, ki_, r=lambda t: 0.55 if t >= 1 else 0.5)
    t3, v3, y3, u3 = speed_run(8, kp_, ki_, d=lambda t: -RAMP if t >= 1 else 0.0, extra=4)
    t4, v4, y4, u4 = speed_run(8, kp_, ki_, d=lambda t: -RAMP if t >= 1 else 0.0)
    rows.append(dict(gains=name, kp=round(kp_, 2), ki=round(ki_, 2), drop=round(0.5 - v.min(), 4),
                     back_in_s=round(t[out[-1]] - 1, 2), pm_deg=round(m["pm"], 1), delay_margin_s=round(m["dm"], 3),
                     step_overshoot_pct=round((v2.max() - 0.55) / 0.05 * 100, 1), step_max_force=round(u2.max(), 2),
                     late_min=round(v3.min(), 3), late_max=round(v3.max(), 3), late_change=round(abs(v3 - v4).max(), 4)))
pd.DataFrame(rows)

1 degree ramp: 1.37 N


,gains,kp,ki,drop,back_in_s,pm_deg,delay_margin_s,step_overshoot_pct,step_max_force,late_min,late_max,late_change
0,RO-060's gains,20.00,10.00,0.0535,5.19,70.0,0.487,6.2,2.01,0.444,0.500,0.0036
1,bump-test rule,30.09,27.89,0.0388,2.28,58.3,0.271,14.6,2.53,0.457,0.500,0.0045
2,Ziegler–Nichols,90.00,270.00,0.0209,0.82,17.9,0.032,83.4,4.00,0.471,0.528,0.0289


## 6. The heading PID: derivative kick and set-point weighting (Section 5)

In [8]:
print("closed-loop poles:", np.round(np.roots([I_YAW, HKD, HKP, HKI]), 4))
rows = []
for beta, gamma in ((1, 1), (1, 0), (0.5, 0), (0, 0)):
    o = heading_run(20, beta=beta, gamma=gamma)
    rows.append(dict(beta=beta, gamma=gamma, request_at_step=round(o["c"][50], 3), max_request=round(o["c"].max(), 3),
                     overshoot_pct=round(max(overshoot(o), 0), 1), within_2pct_after_s=round(settle(o), 2),
                     final=round(o["th"][-1], 4)))
pd.DataFrame(rows)

closed-loop poles: [-1.9266+0.j     -0.4517+0.2356j -0.4517-0.2356j]


,beta,gamma,request_at_step,max_request,overshoot_pct,within_2pct_after_s,final
0,1.0,1,2.85,2.850,21.1,10.97,0.1
1,1.0,0,0.02,0.020,22.5,11.11,0.1
2,0.5,0,0.01,0.010,1.6,5.42,0.1
3,0.0,0,0.00,0.001,0.3,9.53,0.1


## 7. The derivative filter (Section 6.1)

In [9]:
print("tick:", round(DS * 1000, 3), "mm; heading per tick", round(DS / L_WHEELS * 1000, 3), "mrad; D jump",
      round(HKD * DS / L_WHEELS / DT, 4), "N m")
rows = []
for N in (None, 20, 10, 5):
    o = heading_run(2.0, step=0.0, gamma=0.0, quant=True, N=N)
    m = margins(heading_loop_tf(wg, N=N), wg)
    rows.append(dict(N=N or "none", Tf=round(TD / N, 3) if N else 0, D_swing=round(abs(o["D"]).max(), 4),
                     pm_deg=round(m["pm"], 1), true_heading_max_mrad=round(abs(o["th"]).max() * 1000, 2)))
pd.DataFrame(rows)

tick: 0.307 mm; heading per tick 1.534 mrad; D jump 0.0434 N m


,N,Tf,D_swing,pm_deg,true_heading_max_mrad
0,none,0.000,0.0434,75.1,0.72
1,20,0.071,0.0054,64.1,0.71
2,10,0.141,0.0030,53.9,0.72
3,5,0.283,0.0016,38.7,0.73


## 8. Low-pass against notch on the gyro (Section 6.2)

In [10]:
m0 = margins(heading_loop_tf(wg), wg)
print(f"heading loop: crossover {m0['wgc']:.2f} rad/s = {m0['wgc'] / 2 / np.pi:.2f} Hz, phase margin {m0['pm']:.1f} deg")
print("omega0", round(2 * np.pi * F_VIB, 1), "rad/s; Q", F_VIB / 5.0)
rows = []
for name, F in (("none", None), ("low-pass 1 Hz", lowpass(1.0)), ("notch 10 Hz", notch())):
    o = heading_run(2.0, step=0.0, gyro=True, vib=True, filt=F)
    FF = F if F is not None else (np.array([1.0]), np.array([1.0]))
    m = margins(heading_loop_tf(wg, gyro_filt=FF), wg)
    _, h = signal.freqz(*FF, worN=[2 * np.pi * F_VIB, m0["wgc"]], fs=2 * np.pi / DT)
    rows.append(dict(filter=name, gain_10Hz=round(abs(h[0]), 3), lag_at_crossover_deg=round(-np.degrees(np.angle(h[1])), 1),
                     D_swing_after_1_5s=round(abs(o["D"][150:]).max(), 4), pm_deg=round(m["pm"], 1)))
pd.DataFrame(rows)

heading loop: crossover 2.87 rad/s = 0.46 Hz, phase margin 75.1 deg
omega0 62.8 rad/s; Q 2.0


,filter,gain_10Hz,lag_at_crossover_deg,D_swing_after_1_5s,pm_deg
0,none,1.000,-0.0,0.0835,75.1
1,low-pass 1 Hz,0.098,24.4,0.0090,51.6
2,notch 10 Hz,0.000,1.4,0.0006,73.8
